# Libraries:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from tqdm import tqdm
import time
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox
from scipy.stats import zscore, shapiro, jarque_bera
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import mutual_info_regression


import warnings
warnings.filterwarnings("ignore")

# Data Collection:



*   Downloading the Data:




In [ ]:
def get_sp500_tickers():
    """
    Fetches the current list of S&P 500 tickers from Wikipedia.
    Returns:
    - List of tickers with Yahoo-compatible symbols (e.g., BRK-B instead of BRK.B)
    """
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    sp500_table = pd.read_html(url)
    tickers = sp500_table[0]['Symbol'].tolist()
    tickers = [t.replace('.', '-') for t in tickers]
    return tickers

def download_clean_tickers_segmented(tickers, start_date, end_date, min_data_threshold=0.7, batch_size=50):
    """
    Downloads historical Adjusted Close and Volume data for a list of tickers in batches.

    Returns:
    - price_df: DataFrame of adjusted close prices
    - volume_df: DataFrame of volumes
    - failed_tickers: List of failed ticker symbols
    """
    clean_prices = {}
    clean_volumes = {}
    failed_tickers = []

    total_batches = (len(tickers) + batch_size - 1) // batch_size
    print(f"📦 Starting segmented download for {len(tickers)} tickers in {total_batches} batches...\n")

    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i + batch_size]
        print(f"🔹 Batch {i // batch_size + 1}/{total_batches}: Downloading {len(batch)} tickers")

        for ticker in tqdm(batch, desc=f"⏳ Downloading batch {i // batch_size + 1}"):
            try:
                t = yf.Ticker(ticker)
                df = t.history(start=start_date, end=end_date)

                if df.empty or 'Close' not in df.columns:
                    failed_tickers.append(ticker)
                    continue

                price_series = df['Adj Close'] if 'Adj Close' in df.columns else df['Close']
                volume_series = df['Volume'] if 'Volume' in df.columns else pd.Series(index=price_series.index)

                # Filter by completeness
                if price_series.notna().sum() >= min_data_threshold * len(price_series):
                    clean_prices[ticker] = price_series
                    clean_volumes[ticker] = volume_series
                else:
                    failed_tickers.append(ticker)
            except Exception as e:
                print(f"❌ Error downloading {ticker}: {e}")
                failed_tickers.append(ticker)

        # Delay between batches to avoid throttling
        time.sleep(2)

    # Combine to DataFrames
    price_df = pd.DataFrame(clean_prices)
    volume_df = pd.DataFrame(clean_volumes)

    # Remove timezone info
    price_df.index = price_df.index.tz_localize(None)
    volume_df.index = volume_df.index.tz_localize(None)

    print(f"\n✅ Download complete: {price_df.shape[1]} tickers succeeded.")
    print(f"❌ Failed tickers ({len(failed_tickers)}): {failed_tickers}")
    return price_df, volume_df, failed_tickers

# ---------------------------------------------
# 📌 USAGE
# ---------------------------------------------
if __name__ == "__main__":
    # STEP 1: Get Ticker List
    tickers = get_sp500_tickers()

    # STEP 2: Download Data
    price_df, volume_df, failed_tickers = download_clean_tickers_segmented(
        tickers=tickers,
        start_date="1999-11-30",
        end_date="2025-07-01",
        min_data_threshold=0.7,
        batch_size=50
    )

    # STEP 3: Export to Excel
    output_path = 'sp500_prices_volume_segmented.xlsx'
    with pd.ExcelWriter(output_path) as writer:
        price_df.to_excel(writer, sheet_name='Adjusted Close')
        volume_df.to_excel(writer, sheet_name='Volume')

    print(f"\n✅ Excel file saved: {output_path}")


📦 Starting segmented download for 503 tickers in 11 batches...

🔹 Batch 1/11: Downloading 50 tickers


⏳ Downloading batch 1: 100%|██████████| 50/50 [00:24<00:00,  2.07it/s]


🔹 Batch 2/11: Downloading 50 tickers


⏳ Downloading batch 2: 100%|██████████| 50/50 [00:20<00:00,  2.50it/s]


🔹 Batch 3/11: Downloading 50 tickers


...

🔹 Batch 10/11: Downloading 50 tickers


⏳ Downloading batch 10: 100%|██████████| 50/50 [00:22<00:00,  2.18it/s]


🔹 Batch 11/11: Downloading 3 tickers


⏳ Downloading batch 11: 100%|██████████| 3/3 [00:01<00:00,  2.38it/s]



✅ Download complete: 503 tickers succeeded.
❌ Failed tickers (0): []

✅ Excel file saved: sp500_prices_volume_segmented.xlsx




*   Loading the Data:



In [ ]:
# Define file path
file_path = "sp500_prices_volume_segmented.xlsx"

# Read specific sheets
price_df = pd.read_excel(file_path, sheet_name='Adjusted Close', index_col=0, parse_dates=True)
vol_df = pd.read_excel(file_path, sheet_name='Volume', index_col=0, parse_dates=True)

# Preview
print("📊 price_df shape:", price_df.shape)
print("🔊 vol_df shape:", vol_df.shape)

print("\n✅ price_df preview:")
print(price_df.head())

print("\n✅ vol_df preview:")
print(vol_df.head())

📊 price_df shape: (6434, 503)
🔊 vol_df shape: (6434, 503)

✅ price_df preview:
                  MMM       AOS       ABT  ABBV  ACN       ADBE       AMD  \
Date                                                                        
1999-11-30  19.762268  2.310322  8.870531   NaN  NaN  17.044020  14.12500   
1999-12-01  19.529617  2.257815  8.739224   NaN  NaN  16.516727  13.75000   
1999-12-02  20.098314  2.231562  8.772050   NaN  NaN  17.028511  15.25000   
1999-12-03  20.537762  2.165928  8.739224   NaN  NaN  16.547745  15.75000   
1999-12-06  20.227568  2.139673  8.491199   NaN  NaN  15.663751  15.71875   

                  AES       AFL          A  ...       WMB  WTW  WDAY  WYNN  \
Date                                        ...                              
1999-11-30  18.945595  7.171399  25.312435  ...  9.681385  NaN   NaN   NaN   
1999-12-01  18.966032  7.227573  25.762444  ...  9.502101  NaN   NaN   NaN   
1999-12-02  19.109097  7.302471  26.474934  ...  9.735168  NaN   NaN 

# Cleaning the Dataset:

In [ ]:
# Define a cleaning function
def clean_dataframe(df, name=''):
    print(f"🔧 Cleaning {name}...")

    # Step 1: Fill missing values from both ends
    df = df.bfill().ffill()

    # Step 2: Linear interpolation for smoother adjustment
    df = df.interpolate(method='linear', axis=0)

    # Step 3: Final removal of any remaining NaNs
    df = df.dropna()

    print(f"✅ {name} cleaned. Final shape: {df.shape}\n")
    return df

# Clean both price and volume DataFrames
price_df_clean = clean_dataframe(price_df, name="Price Data")
vol_df_clean = clean_dataframe(vol_df, name="Volume Data")

# Save to Excel with two separate sheets
output_path = "SP500_Cleaned_Data.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    price_df_clean.to_excel(writer, sheet_name='Prices', index=True)
    vol_df_clean.to_excel(writer, sheet_name='Volumes', index=True)

print(f"📁 Data saved successfully to: {output_path}")

🔧 Cleaning Price Data...
✅ Price Data cleaned. Final shape: (6434, 503)

🔧 Cleaning Volume Data...
✅ Volume Data cleaned. Final shape: (6434, 503)

📁 Data saved successfully to: SP500_Cleaned_Data.xlsx


# Top 200 Tickers:


In [ ]:
#Create 1-Month Forward Return Target
future_return_1m = price_df_clean.pct_change(periods=21).shift(-21)

#Generate Basic Features
returns = price_df_clean.pct_change().dropna()
log_returns = np.log(price_df_clean / price_df_clean.shift(1)).dropna()
sma_10 = price_df_clean.rolling(window=10).mean()
sma_50 = price_df_clean.rolling(window=50).mean()
ema_10 = price_df_clean.ewm(span=10).mean()
volatility_21 = returns.rolling(21).std()

#Stack into Panel Format
features_dict = {
    'return_1d': returns,
    'log_return': log_returns,
    'sma_10': sma_10,
    'sma_50': sma_50,
    'ema_10': ema_10,
    'vol_21d': volatility_21
}

feature_frames = []
for name, df in features_dict.items():
    df_long = df.stack().rename(name)
    feature_frames.append(df_long)

features_panel = pd.concat(feature_frames, axis=1)
target_panel = future_return_1m.stack().rename("target_1m")
final_df = pd.concat([features_panel, target_panel], axis=1).dropna()

#Score Tickers by Predictive Power
from sklearn.feature_selection import mutual_info_regression

mi_scores = {}
for ticker in final_df.index.get_level_values(1).unique():
    df_ticker = final_df.xs(ticker, level=1)
    if df_ticker.shape[0] > 100:
        X = df_ticker.drop(columns=['target_1m'])
        y = df_ticker['target_1m']
        mi = mutual_info_regression(X, y)
        mi_scores[ticker] = np.mean(mi)

mi_series = pd.Series(mi_scores).sort_values(ascending=False)
top200_tickers = mi_series.head(200).index.tolist()

#Subset the Data
price_df_top200 = price_df_clean[top200_tickers]
vol_df_top200 = vol_df_clean[top200_tickers]

#Save Final Output
output_path = "SP500_Top200_Featured.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    price_df_top200.to_excel(writer, sheet_name='Prices')
    vol_df_top200.to_excel(writer, sheet_name='Volumes')

print(f"📁 Top 200 data saved to: {output_path}")



📁 Top 200 data saved to: SP500_Top200_Featured.xlsx


In [ ]:
# --- Drop-in: robust HTTPS downloader for Goyal & Welch (monthly predictors) ---

import time, requests, pandas as pd, os

GW_BASES = [
    "https://www.hec.unil.ch/agoyal/docs/",   # try HTTPS first (more reliable)
    "http://www.hec.unil.ch/agoyal/docs/",    # fallback to HTTP
]
GW_NAMES = [f"PredictorData{y}.xlsx" for y in range(2025, 2017, -1)]  # 2025 → 2018
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

def _retry_session(retries=5, backoff=1.5, timeout=120):
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (compatible; thesis-bot/1.0)",
        "Accept": "*/*",
        "Connection": "keep-alive"
    })
    s.retries = retries
    s.backoff = backoff
    s.default_timeout = timeout
    return s

def _get_with_retries(session, url, timeout=120, max_attempts=5, backoff=1.7, chunk_bytes=1<<20):
    last_err = None
    for attempt in range(1, max_attempts+1):
        try:
            with session.get(url, timeout=timeout, stream=True) as r:
                r.raise_for_status()
                # stream to disk to avoid memory issues
                tmp_path = os.path.join(DATA_DIR, "_tmp_gw_download.bin")
                with open(tmp_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=chunk_bytes):
                        if chunk:
                            f.write(chunk)
                return tmp_path
        except Exception as e:
            last_err = e
            sleep_s = backoff ** attempt
            print(f"  attempt {attempt}/{max_attempts} failed for {url}: {e} — retrying in {sleep_s:.1f}s")
            time.sleep(sleep_s)
    raise last_err

def try_get_goyal_welch():
    sess = _retry_session()
    for name in GW_NAMES:
        for base in GW_BASES:
            url = base + name
            try:
                print(f"Trying {url}")
                tmp_path = _get_with_retries(sess, url)
                # save as stable name
                xlsx_path = os.path.join(DATA_DIR, "goyal_welch_predictors.xlsx")
                os.replace(tmp_path, xlsx_path)

                # parse a likely monthly sheet; fallback to first sheet
                xls = pd.ExcelFile(xlsx_path)
                sheet = next((s for s in xls.sheet_names if "Monthly" in s or "monthly" in s), xls.sheet_names[0])
                df = pd.read_excel(xlsx_path, sheet_name=sheet, header=0)

                # detect date column
                cand = next((c for c in df.columns if str(c).lower() in {"yyyymm","date","yearmon","year_month"}), df.columns[0])
                ser = df[cand]

                # build monthly Date index
                if pd.api.types.is_numeric_dtype(ser):
                    dt = pd.to_datetime(ser.astype(int).astype(str), format="%Y%m", errors="coerce")
                else:
                    dt = pd.to_datetime(ser, errors="coerce")
                    # normalize to YYYY-MM month-end
                    dt = dt.dt.to_period("M").dt.to_timestamp("M")

                df = df.loc[dt.notna()].copy()
                df["Date"] = dt.loc[dt.notna()]
                df = df.set_index("Date").sort_index().dropna(how="all").dropna(axis=1, how="all")

                out_csv = os.path.join(DATA_DIR, "goyal_welch_predictors.csv")
                df.to_csv(out_csv)
                print(f"Saved {out_csv} (rows={len(df):,}, cols={len(df.columns):,}) from {name}")
                return df
            except Exception as e:
                print(f"Skip {url}: {e}")
    print("❗Could not fetch Goyal & Welch automatically. Manual download: https://www.hec.unil.ch/agoyal/")
    return None

# --- helper to align monthly predictors to daily index (e.g., your prices) ---
def align_monthly_to_daily(monthly_df: pd.DataFrame, daily_index: pd.DatetimeIndex) -> pd.DataFrame:
    """
    Forward-fill monthly values onto a daily index (end-of-month stamped to all following days until next EOM).
    """
    m = monthly_df.asfreq("M")  # ensure monthly freq at month-end
    daily = m.reindex(pd.date_range(daily_index.min(), daily_index.max(), freq="D")).ffill()
    return daily.reindex(daily_index)


1) Imports & Constants

In [ ]:
# --- Cell 1: imports & constants ---
import os, io, zipfile, requests, pandas as pd

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

FF_URL = {
    ("daily","3f"): "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip",
    ("monthly","3f"): "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip",
    ("daily","5f"): "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip",
}

# Goyal & Welch (simple HTTP; see robust option in Cell 6 if needed)
GW_BASE = "http://www.hec.unil.ch/agoyal/docs/"
GW_NAMES = [f"PredictorData{y}.xlsx" for y in range(2025, 2017, -1)]


2) Core Helpers (download, unzip, FF parser)

In [ ]:
# --- Cell 2: helpers for download/parsing ---

def download(url: str) -> bytes:
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.content

def extract_first_csv_from_zip(zbytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zbytes)) as zf:
        csvs = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if not csvs:
            raise ValueError("ZIP has no CSV inside.")
        return zf.read(csvs[0]).decode("latin1")

def parse_ff(raw: str, freq: str) -> pd.DataFrame:
    # find header line containing factor names
    lines = raw.splitlines()
    hdr = next(i for i, l in enumerate(lines) if "Mkt-RF" in l and "RF" in l)
    foot = next((i for i, l in enumerate(lines[hdr+1:], hdr+1)
                 if "Annual" in l or "Copyright" in l), len(lines))
    df = pd.read_csv(io.StringIO("\n".join(lines[hdr:foot])))
    df.columns = [c.strip().replace(" ", "") for c in df.columns]
    if df.columns[0].lower() != "date":
        df = df.rename(columns={df.columns[0]: "Date"})
    s = df["Date"].astype(str).str.strip()
    need = 8 if freq == "daily" else 6
    df = df[s.str.fullmatch(rf"\d{{{need}}}")].copy()
    if freq == "daily":
        df["Date"] = pd.to_datetime(df["Date"], format="%Y%m%d")
    else:
        df["Date"] = pd.to_datetime(df["Date"], format="%Y%m") + pd.offsets.MonthEnd(0)
    df = df.set_index("Date").sort_index()
    pct_cols = [c for c in ["Mkt-RF","SMB","HML","RMW","CMA","RF"] if c in df.columns]
    df[pct_cols] = df[pct_cols].apply(pd.to_numeric, errors="coerce") / 100.0
    return df


3) Fama–French Downloaders (daily/monthly, 3F/5F)


In [ ]:
# --- Cell 3: Fama–French getters ---

def get_ff(freq: str, kind: str) -> pd.DataFrame:
    raw = extract_first_csv_from_zip(download(FF_URL[(freq, kind)]))
    df = parse_ff(raw, freq)
    out = os.path.join(DATA_DIR, f"fama_french_{kind}_{freq}.csv")
    df.to_csv(out)
    print(f"Saved {out} ({len(df)} rows, {len(df.columns)} cols)")
    return df


4) Goyal & Welch (simple version) + Monthly→Daily aligner

In [ ]:
# --- Cell 4: Goyal & Welch (simple HTTP) + monthly→daily aligner ---

def try_goyal_welch_simple() -> pd.DataFrame | None:
    for name in GW_NAMES:
        try:
            b = download(GW_BASE + name)
            xlsx = os.path.join(DATA_DIR, "goyal_welch.xlsx")
            with open(xlsx, "wb") as f:
                f.write(b)
            xls = pd.ExcelFile(xlsx)
            sheet = next((s for s in xls.sheet_names if "Monthly" in s or "monthly" in s), xls.sheet_names[0])
            df = pd.read_excel(xlsx, sheet_name=sheet)

            cand = next((c for c in df.columns if str(c).lower() in {"yyyymm","date","yearmon","year_month"}), df.columns[0])
            ser = df[cand]
            if pd.api.types.is_numeric_dtype(ser):
                dt = pd.to_datetime(ser.astype(int).astype(str), format="%Y%m", errors="coerce")
            else:
                dt = pd.to_datetime(ser, errors="coerce")
                dt = dt.dt.to_period("M").dt.to_timestamp("M")

            df = df.loc[dt.notna()].copy()
            df["Date"] = dt.loc[dt.notna()]
            df = df.set_index("Date").sort_index().dropna(how="all").dropna(axis=1, how="all")

            out = os.path.join(DATA_DIR, "goyal_welch.csv")
            df.to_csv(out)
            print(f"Saved {out} ({len(df)} rows, {len(df.columns)} cols) from {name}")
            return df
        except Exception as e:
            print(f"Skip {name}: {e}")
    print("Goyal–Welch auto-download failed; get manually from https://www.hec.unil.ch/agoyal/")
    return None

def align_monthly_to_daily(monthly_df: pd.DataFrame, daily_index: pd.DatetimeIndex) -> pd.DataFrame:
    """Forward-fill monthly series to a given daily index."""
    m = monthly_df.asfreq("M")
    daily_full = m.reindex(pd.date_range(daily_index.min(), daily_index.max(), freq="D")).ffill()
    return daily_full.reindex(daily_index)


5) Example Runs (FF 3F/5F) + Merge RF with your prices

In [ ]:
# --- Cell 5: run FF downloads + (optional) merge RF with your prices ---

ff_d3 = get_ff("daily", "3f")
ff_m3 = get_ff("monthly", "3f")

# 5-factor daily is optional; some regions throttle
try:
    ff_d5 = get_ff("daily", "5f")
except Exception as e:
    print("5F daily failed (optional):", e)

# Example: merge daily RF into your cleaned prices (uncomment and adapt path)
# prices = pd.read_csv("SP500_Prices_Clean.csv", parse_dates=["Date"], index_col="Date")
# merged = prices.join(ff_d3[["RF"]], how="left")
# merged["RF"] = merged["RF"].ffill()
# merged.to_csv(os.path.join(DATA_DIR, "prices_with_rf.csv"))
# print("Saved data/prices_with_rf.csv")


Saved data/fama_french_3f_daily.csv (26023 rows, 4 cols)
Saved data/fama_french_3f_monthly.csv (1188 rows, 4 cols)
Saved data/fama_french_5f_daily.csv (15603 rows, 6 cols)


6) Goyal & Welch (robust HTTPS with retries & backoff) — use this if the simple one times out

In [ ]:
# --- Cell 6: Robust Goyal & Welch (HTTPS + retries + streaming) ---

import time

GW_BASES = [
    "https://www.hec.unil.ch/agoyal/docs/",
    "http://www.hec.unil.ch/agoyal/docs/",
]

def _retry_session(retries=5, timeout=120):
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0 (thesis-bot)"})
    s.timeout = timeout
    s.retries = retries
    return s

def _get_stream(session, url, timeout=120, max_attempts=5, backoff=1.7, chunk=1<<20):
    last = None
    for k in range(1, max_attempts+1):
        try:
            with session.get(url, timeout=timeout, stream=True) as r:
                r.raise_for_status()
                tmp = os.path.join(DATA_DIR, "_gw_tmp.bin")
                with open(tmp, "wb") as f:
                    for c in r.iter_content(chunk_size=chunk):
                        if c: f.write(c)
                return tmp
        except Exception as e:
            last = e
            wait = backoff ** k
            print(f"  attempt {k}/{max_attempts} failed for {url}: {e} — retry in {wait:.1f}s")
            time.sleep(wait)
    raise last

def try_goyal_welch_robust() -> pd.DataFrame | None:
    sess = _retry_session()
    for name in GW_NAMES:
        for base in GW_BASES:
            url = base + name
            try:
                print(f"Trying {url}")
                tmp = _get_stream(sess, url)
                xlsx = os.path.join(DATA_DIR, "goyal_welch_predictors.xlsx")
                os.replace(tmp, xlsx)

                xls = pd.ExcelFile(xlsx)
                sheet = next((s for s in xls.sheet_names if "Monthly" in s or "monthly" in s), xls.sheet_names[0])
                df = pd.read_excel(xlsx, sheet_name=sheet)

                cand = next((c for c in df.columns if str(c).lower() in {"yyyymm","date","yearmon","year_month"}), df.columns[0])
                ser = df[cand]
                if pd.api.types.is_numeric_dtype(ser):
                    dt = pd.to_datetime(ser.astype(int).astype(str), format="%Y%m", errors="coerce")
                else:
                    dt = pd.to_datetime(ser, errors="coerce").dt.to_period("M").dt.to_timestamp("M")

                df = df.loc[dt.notna()].copy()
                df["Date"] = dt.loc[dt.notna()]
                df = df.set_index("Date").sort_index().dropna(how="all").dropna(axis=1, how="all")

                out = os.path.join(DATA_DIR, "goyal_welch_predictors.csv")
                df.to_csv(out)
                print(f"Saved {out} ({len(df)} rows, {len(df.columns)} cols) from {name}")
                return df
            except Exception as e:
                print(f"Skip {url}: {e}")
    print("❗Could not fetch Goyal & Welch automatically. Manual: https://www.hec.unil.ch/agoyal/")
    return None


7) Example: Run Goyal & Welch + Align to Daily Prices

In [ ]:
# --- Cell B1: imports, constants, helper to fetch FRED CSV ---

import os, io, requests
import pandas as pd

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

def fred_csv(series: str, timeout=120) -> pd.DataFrame:
    """
    Pulls a single FRED series as a DataFrame with a Date index.
    No API key needed (uses public CSV endpoint).
    """
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}"
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    # standardize
    if "DATE" not in df.columns:
        raise ValueError(f"Unexpected FRED CSV format for {series}")
    df["Date"] = pd.to_datetime(df["DATE"])
    df = df.drop(columns=["DATE"]).rename(columns={series: series})
    df[series] = pd.to_numeric(df[series], errors="coerce")
    return df.set_index("Date").sort_index()


In [ ]:
import pandas as pd, requests, io, os

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

url = "https://www.ivo-welch.info/professional/goyal-welch/goyal-welch-a.csv"
r = requests.get(url, timeout=120)
r.raise_for_status()

# Use io.StringIO to treat text as file-like
df = pd.read_csv(io.StringIO(r.text))

out_path = os.path.join(DATA_DIR, "gw_welch_mirror.csv")
df.to_csv(out_path, index=False)

print("Saved", out_path, "shape:", df.shape)
df.head()
df

Saved data/gw_welch_mirror.csv shape: (219, 24)


,yyyy,cpi,gold,infl,tbill,ltyld10,ltrate,callmoney,aaa,baa,...,sp500e12,vwm,vwmx,svar,bkmk,ntis,eqis,csp,cay,ik
0,1802,0.936,19.39,-15.730,NaN,6.60,NaN,NaN,NaN,NaN,...,NaN,12.140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1803,0.788,19.39,5.490,NaN,6.76,NaN,NaN,NaN,NaN,...,NaN,-4.780,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1804,0.832,19.39,4.380,NaN,6.90,NaN,NaN,NaN,NaN,...,NaN,0.780,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1805,0.868,19.39,-0.700,NaN,7.00,NaN,NaN,NaN,NaN,...,NaN,0.610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1806,0.862,19.39,4.220,NaN,6.73,NaN,NaN,NaN,NaN,...,NaN,10.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214,2016,18.885,1250.74,2.075,0.51,2.72,1.761,NaN,4.06,4.83,...,94.55,11.793,9.330,1.712,29.347910,-2.48100,8.954,NaN,-5.640,3.181
215,2017,19.283,1257.12,2.109,1.32,2.54,6.230,NaN,3.51,4.22,...,109.88,21.989,19.531,0.454,23.539334,-1.99500,7.746,NaN,-3.163,3.314
216,2018,19.650,1268.49,1.910,2.37,2.84,-0.006,NaN,4.02,5.13,...,132.39,-4.492,-6.354,2.890,26.808877,-1.92000,8.110,NaN,-2.890,3.490
217,2019,20.100,1392.60,2.285,1.54,1.90,12.165,NaN,3.01,3.88,...,139.47,31.500,28.900,1.580,22.994390,-0.73000,9.390,NaN,-4.160,3.350


In [ ]:
# Path B · Cell 1: imports, dirs, robust FRED fetcher

import os, io, time, requests
import pandas as pd

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

def fred_csv(series: str, timeout=120, max_attempts=5, backoff=1.7) -> pd.DataFrame:
    """
    Fetch a single FRED series as a DataFrame with Date index.
    Tries primary CSV endpoint with retries. Falls back to flexible parsing.
    """
    def _get(url):
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        txt = r.text
        if "<html" in txt.lower() or "<!doctype html" in txt.lower():
            raise ValueError("FRED returned HTML (rate-limited / outage).")
        return txt

    primary = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}"
    alt     = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}&cosd=1950-01-01"

    last_err = None
    for attempt in range(1, max_attempts+1):
        try:
            try:
                txt = _get(primary)
            except Exception:
                txt = _get(alt)
            df = pd.read_csv(io.StringIO(txt))
            if "DATE" not in df.columns:
                df.rename(columns={df.columns[0]: "DATE"}, inplace=True)
            if series not in df.columns and len(df.columns) >= 2:
                df.rename(columns={df.columns[1]: series}, inplace=True)
            df["Date"] = pd.to_datetime(df["DATE"], errors="coerce")
            df = df.dropna(subset=["Date"]).drop(columns=["DATE"]).set_index("Date").sort_index()
            df[series] = pd.to_numeric(df[series], errors="coerce")
            return df
        except Exception as e:
            last_err = e
            time.sleep(backoff ** attempt)
    raise last_err


In [ ]:
# Path B · Cell 2: pull core predictors from FRED and resample to month-end

series = ["DGS10", "TB3MS", "BAA", "AAA", "CPIAUCSL", "INDPRO", "UNRATE"]
dfs = []
for s in series:
    df_s = fred_csv(s).resample("ME").last()
    dfs.append(df_s)
    time.sleep(0.6)  # gentler on FRED

gw_lite_m = pd.concat(dfs, axis=1)

# Spreads / transforms (match GW spirit)
gw_lite_m["TERM"]    = gw_lite_m["DGS10"] - gw_lite_m["TB3MS"]      # 10Y - 3M
gw_lite_m["DEF"]     = gw_lite_m["BAA"]   - gw_lite_m["AAA"]        # BAA - AAA
gw_lite_m["INF_YoY"] = gw_lite_m["CPIAUCSL"].pct_change(12) * 100.0 # CPI YoY %
gw_lite_m["IP_YoY"]  = gw_lite_m["INDPRO"].pct_change(12)  * 100.0  # IP YoY %

# Tidy up
gw_lite_m = gw_lite_m.dropna(how="all")

out_csv = os.path.join(DATA_DIR, "gw_lite_monthly.csv")
gw_lite_m.to_csv(out_csv)
print(f"Saved {out_csv}  rows={len(gw_lite_m):,}  cols={len(gw_lite_m.columns):,}")
gw_lite_m.tail()


/tmp/ipython-input-3702699266.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_s = fred_csv(s).resample("M").last()
/tmp/ipython-input-3702699266.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_s = fred_csv(s).resample("M").last()
/tmp/ipython-input-3702699266.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_s = fred_csv(s).resample("M").last()
/tmp/ipython-input-3702699266.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_s = fred_csv(s).resample("M").last()
/tmp/ipython-input-3702699266.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_s = fred_csv(s).resample("M").last()
/tmp/ipython-input-3702699266.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, pleas

Saved data/gw_lite_monthly.csv  rows=1,280  cols=11


/tmp/ipython-input-3702699266.py:15: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  gw_lite_m["INF_YoY"] = gw_lite_m["CPIAUCSL"].pct_change(12) * 100.0 # CPI YoY %
/tmp/ipython-input-3702699266.py:16: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  gw_lite_m["IP_YoY"]  = gw_lite_m["INDPRO"].pct_change(12)  * 100.0  # IP YoY %


,DGS10,TB3MS,BAA,AAA,CPIAUCSL,INDPRO,UNRATE,TERM,DEF,INF_YoY,IP_YoY
Date,,,,,,,,,,,
2025-04-30,4.17,4.21,6.18,5.45,320.321,103.6696,4.2,-0.04,0.73,2.333747,1.282572
2025-05-31,4.41,4.25,6.29,5.54,320.580,103.7484,4.2,0.16,0.75,2.375934,0.746458
2025-06-30,4.24,4.23,6.15,5.46,321.500,104.1137,4.1,0.01,0.69,2.672683,0.833193
2025-07-31,4.37,4.25,6.10,5.45,322.132,103.9867,4.2,0.12,0.65,2.731801,1.431439
2025-08-31,4.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.547027,0.938753


In [ ]:
# Path B · Cell 3: optional extras (add if you want more signals)

extras = ["T10Y2Y", "TEDRATE", "M2SL", "VIXCLS"]  # 10y-2y, TED spread, money supply, VIX
for s in extras:
    try:
        df_s = fred_csv(s).resample("M").last()
        gw_lite_m = gw_lite_m.join(df_s, how="outer")
        time.sleep(0.6)
    except Exception as e:
        print(f"Skip {s}: {e}")

gw_lite_m.to_csv(os.path.join(DATA_DIR, "gw_lite_monthly.csv"))
print("Re-saved gw_lite_monthly.csv", gw_lite_m.shape)


/tmp/ipython-input-2743976977.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_s = fred_csv(s).resample("M").last()
/tmp/ipython-input-2743976977.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_s = fred_csv(s).resample("M").last()
/tmp/ipython-input-2743976977.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_s = fred_csv(s).resample("M").last()
/tmp/ipython-input-2743976977.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_s = fred_csv(s).resample("M").last()


Re-saved gw_lite_monthly.csv (1280, 15)


In [ ]:
# Path B · Cell 4: forward-fill monthly values onto a daily index

def align_monthly_to_daily(monthly_df: pd.DataFrame, daily_index: pd.DatetimeIndex) -> pd.DataFrame:
    """
    Forward-fill month-end values across days until the next month-end, then reindex to your daily dates.
    """
    m = monthly_df.asfreq("M")
    daily_full = m.reindex(pd.date_range(daily_index.min(), daily_index.max(), freq="D")).ffill()
    return daily_full.reindex(daily_index)


In [ ]:
import os
out_xlsx = os.path.join(DATA_DIR, "gw_lite_monthly.xlsx")

# Save the predictors as Excel
gw_lite_m.to_excel(out_xlsx)

print(f"Saved GW-lite predictors to {out_xlsx}")
print("Shape:", gw_lite_m.shape)
print("Columns:", gw_lite_m.columns.tolist())


Saved GW-lite predictors to data/gw_lite_monthly.xlsx
Shape: (1280, 15)
Columns: ['DGS10', 'TB3MS', 'BAA', 'AAA', 'CPIAUCSL', 'INDPRO', 'UNRATE', 'TERM', 'DEF', 'INF_YoY', 'IP_YoY', 'T10Y2Y', 'TEDRATE', 'M2SL', 'VIXCLS']


In [1]:
#!/usr/bin/env python3
"""
Download Market Data (Simplified & Robust)
------------------------------------------
What this script does (consolidated from your notebook):
  • Pull S&P 500 tickers (Wikipedia), normalize for Yahoo (BRK.B → BRK-B)
  • Download Adjusted Close & Volume via yfinance, in batches with retries
  • Clean data (bfill/ffill + linear interpolate → drop remaining NaN rows)
  • Save Prices and Volumes into ONE Excel file with two sheets
  • Download Fama–French 3 Factors (daily & monthly) from Ken French
  • Build a "GW-lite" macro panel from FRED: TERM, DEF, CPI YoY, IP YoY (+ optional extras)
  • Save everything under ./data/
  • Provide a helper to forward-fill monthly series onto daily trading dates

Usage (example):
  python download_market_data.py --start 2010-01-01 --end 2025-06-30 --outfile SP500_PricesVolumes.xlsx

Dependencies:
  pip install pandas numpy yfinance requests openpyxl
"""

import os, io, time, math, argparse, sys
from typing import List, Tuple
import pandas as pd
import numpy as np
import requests

# ------------------------------
# Config
# ------------------------------
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# ------------------------------
# Utils
# ------------------------------
def log(msg: str):
    print(f"[INFO] {msg}", flush=True)

def warn(msg: str):
    print(f"[WARN] {msg}", flush=True)

def err(msg: str):
    print(f"[ERROR] {msg}", flush=True)


# ------------------------------
# S&P 500 tickers (Wikipedia)
# ------------------------------
def get_sp500_tickers() -> List[str]:
    """Fetch current S&P 500 tickers from Wikipedia and convert '.' to '-' for Yahoo compatibility."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    tables = pd.read_html(url)
    tickers = tables[0]["Symbol"].astype(str).str.strip().tolist()
    tickers = [t.replace(".", "-") for t in tickers]
    return tickers


# ------------------------------
# yfinance download (batched)
# ------------------------------
def _safe_import_yf():
    try:
        import yfinance as yf  # type: ignore
    except Exception as e:
        err("yfinance is not installed. Please: pip install yfinance")
        raise
    return yf

def _download_batch(yf, batch: List[str], start: str, end: str, tries=4, sleep=2.0) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Download a batch of tickers. Returns (Adj Close, Volume)."""
    last_exc = None
    for k in range(1, tries+1):
        try:
            log(f"Downloading batch of {len(batch)} tickers (attempt {k}/{tries})")
            data = yf.download(batch, start=start, end=end, auto_adjust=False, progress=False, group_by='column', threads=True)
            # yfinance returns a multi-index column (field first). We want Adj Close & Volume.
            if isinstance(data.columns, pd.MultiIndex):
                adj = data["Adj Close"].copy()
                vol = data["Volume"].copy()
            else:
                # Single ticker case
                adj = data.rename(columns={"Adj Close": batch[0]}).filter(items=["Adj Close"])
                vol = data.rename(columns={"Volume": batch[0]}).filter(items=["Volume"])
            return adj, vol
        except Exception as e:
            last_exc = e
            wait = sleep * (1.6 ** (k-1))
            warn(f"Batch download failed ({e}); retrying in {wait:.1f}s ...")
            time.sleep(wait)
    raise last_exc

def download_sp500_prices_volumes(start: str, end: str, batch_size=50) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    """Download Adj Close & Volume for S&P500 in batches with retries. Returns (prices, volumes, failed)."""
    yf = _safe_import_yf()
    tickers = get_sp500_tickers()
    log(f"Found {len(tickers)} tickers from Wikipedia.")
    prices_list, vols_list = [], []
    failed = []

    total_batches = math.ceil(len(tickers)/batch_size)
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        try:
            adj, vol = _download_batch(yf, batch, start, end)
            prices_list.append(adj)
            vols_list.append(vol)
            time.sleep(0.8)  # be polite
        except Exception as e:
            warn(f"Batch failed for tickers[{i}:{i+batch_size}]: {e}")
            failed.extend(batch)

    price_df = pd.concat(prices_list, axis=1).sort_index()
    vol_df   = pd.concat(vols_list, axis=1).sort_index()

    # sanity: drop duplicate columns just in case
    price_df = price_df.loc[:, ~price_df.columns.duplicated()]
    vol_df   = vol_df.loc[:, ~vol_df.columns.duplicated()]

    return price_df, vol_df, failed


# ------------------------------
# Cleaning
# ------------------------------
def clean_dataframe(df: pd.DataFrame, name="") -> pd.DataFrame:
    log(f"Cleaning {name} ...")
    # bfill/ffill, then gentle linear interpolate, then drop any remaining all-NaN rows.
    out = df.bfill().ffill().interpolate(method="linear", axis=0).dropna(how="any")
    log(f"{name} cleaned → shape={out.shape}")
    return out


# ------------------------------
# Save to Excel (two sheets)
# ------------------------------
def save_prices_volumes_excel(price_df: pd.DataFrame, vol_df: pd.DataFrame, out_path: str):
    log(f"Saving Excel to {out_path} ...")
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        price_df.to_excel(writer, sheet_name="Prices")
        vol_df.to_excel(writer, sheet_name="Volumes")
    log("Excel saved.")


# ------------------------------
# Fama–French 3 Factors (Ken French)
# ------------------------------
FF_URL = {
    ("daily","3f"): "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip",
    ("monthly","3f"): "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip",
}

def _download_bytes(url: str, timeout=120) -> bytes:
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.content

def _extract_first_csv_from_zip(zbytes: bytes) -> str:
    import zipfile
    with zipfile.ZipFile(io.BytesIO(zbytes)) as zf:
        csvs = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if not csvs:
            raise ValueError("ZIP from Ken French contains no CSV.")
        return zf.read(csvs[0]).decode("latin1")

def _parse_ff(raw: str, freq: str) -> pd.DataFrame:
    lines = raw.splitlines()
    # find the header line that contains factor names
    hdr = next(i for i, l in enumerate(lines) if "Mkt-RF" in l and "RF" in l)
    # find footer before "Copyright" or "Annual"
    foot = next((i for i, l in enumerate(lines[hdr+1:], hdr+1) if "Annual" in l or "Copyright" in l), len(lines))
    df = pd.read_csv(io.StringIO("\n".join(lines[hdr:foot])))
    df.columns = [c.strip().replace(" ", "") for c in df.columns]
    if df.columns[0].lower() != "date":
        df = df.rename(columns={df.columns[0]: "Date"})
    s = df["Date"].astype(str).str.strip()
    need = 8 if freq == "daily" else 6
    df = df[s.str.fullmatch(rf"\\d{{{need}}}")].copy()

    if freq == "daily":
        df["Date"] = pd.to_datetime(df["Date"], format="%Y%m%d")
    else:
        df["Date"] = pd.to_datetime(df["Date"], format="%Y%m")
        df["Date"] = df["Date"].dt.to_period("M").dt.to_timestamp("M")

    df = df.set_index("Date").sort_index()
    # convert numeric columns
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def get_fama_french(freq: str) -> pd.DataFrame:
    raw = _extract_first_csv_from_zip(_download_bytes(FF_URL[(freq, "3f")]))
    df = _parse_ff(raw, freq)
    out = os.path.join(DATA_DIR, f"fama_french_3f_{freq}.csv")
    df.to_csv(out)
    log(f"Saved {out} (rows={len(df):,}, cols={len(df.columns)})")
    return df


# ------------------------------
# FRED helpers (GW-lite)
# ------------------------------
def fred_csv(series: str, timeout=120) -> pd.DataFrame:
    """Fetch a single FRED series as DataFrame with Date index (public CSV endpoint)."""
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}"
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    if "DATE" not in df.columns:
        raise ValueError(f"Unexpected FRED CSV format for {series}")
    df["Date"] = pd.to_datetime(df["DATE"])
    df = df.drop(columns=["DATE"]).rename(columns={series: series})
    df[series] = pd.to_numeric(df[series], errors="coerce")
    return df.set_index("Date").sort_index()


def build_gw_lite() -> pd.DataFrame:
    """Build a lighter, reproducible subset of GW predictors from FRED."""
    series = ["DGS10", "TB3MS", "BAA", "AAA", "CPIAUCSL", "INDPRO", "UNRATE"]
    dfs = []
    for s in series:
        df_s = fred_csv(s).resample("ME").last()
        dfs.append(df_s)
        time.sleep(0.5)  # gentle on FRED
    gw = pd.concat(dfs, axis=1)

    # Derived predictors (GW spirit)
    gw["TERM"]    = gw["DGS10"] - gw["TB3MS"]      # 10Y - 3M
    gw["DEF"]     = gw["BAA"]   - gw["AAA"]        # BAA - AAA
    gw["INF_YoY"] = gw["CPIAUCSL"].pct_change(12) * 100.0
    gw["IP_YoY"]  = gw["INDPRO"].pct_change(12) * 100.0

    gw = gw.dropna(how="all")
    out_csv = os.path.join(DATA_DIR, "gw_lite_monthly.csv")
    gw.to_csv(out_csv)
    log(f"Saved GW-lite to {out_csv}  rows={len(gw):,} cols={len(gw.columns):,}")
    return gw


# ------------------------------
# Monthly → Daily aligner
# ------------------------------
def align_monthly_to_daily(monthly_df: pd.DataFrame, daily_index: pd.DatetimeIndex) -> pd.DataFrame:
    """Forward-fill month-end values across a daily index (e.g., trading dates)."""
    m = monthly_df.asfreq("M")
    daily_full = m.reindex(pd.date_range(daily_index.min(), daily_index.max(), freq="D")).ffill()
    return daily_full.reindex(daily_index)


# ------------------------------
# Main
# ------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--start", required=True, help="YYYY-MM-DD")
    ap.add_argument("--end",   required=True, help="YYYY-MM-DD")
    ap.add_argument("--outfile", default="SP500_PricesVolumes.xlsx")
    args = ap.parse_args()

    # 1) S&P 500 prices & volumes
    price_df, vol_df, failed = download_sp500_prices_volumes(args.start, args.end, batch_size=50)
    if failed:
        warn(f"Failed tickers count: {len(failed)} — see stderr list below")
        for t in failed:
            sys.stderr.write(t + "\\n")

    # 2) Clean
    price_clean = clean_dataframe(price_df, "Prices")
    vol_clean   = clean_dataframe(vol_df,   "Volumes")

    # 3) Save Prices/Volumes (two sheets)
    save_prices_volumes_excel(price_clean, vol_clean, args.outfile)

    # 4) Fama–French (daily + monthly 3F)
    ff_daily   = get_fama_french("daily")
    ff_monthly = get_fama_french("monthly")

    # 5) GW-lite (FRED)
    gw_lite = build_gw_lite()

    # 6) Save a merged "prices + RF" example (optional)
    try:
        merged = price_clean.join(ff_daily[["RF"]], how="left")
        merged["RF"] = merged["RF"].ffill()
        merged.to_csv(os.path.join(DATA_DIR, "prices_with_rf.csv"))
        log("Saved data/prices_with_rf.csv")
    except Exception as e:
        warn(f"Could not save prices_with_rf.csv: {e}")

    log("All done.")

if __name__ == "__main__":
    main()


usage: colab_kernel_launcher.py [-h] --start START --end END
                                [--outfile OUTFILE]
colab_kernel_launcher.py: error: the following arguments are required: --start, --end
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --start, --end

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-225270545.py", line 294, in <cell line: 0>
    main()
  File "/tmp/ipython-input-225270545.py", line 259, in main
    args = ap.parse_args()
           ^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 1904, 

TypeError: object of type 'NoneType' has no len()